# 🛡️ GSL Modo 1 — Observación Pasiva
**Geometric Signature Layer — Capa de Firma Geométrica**

## Principio de operación
Este notebook **solo observa**. No bloquea, no alerta, no interviene en ningún sistema del cliente.
Construye el manifold geométrico de lo normal para cada extremo registrado y produce un
score de disonancia continuo que el cliente puede revisar en su panel.

## Arquitectura de nodos
```
Principal (empresa)
  └── Embudo universal (Módulo Maestro)
  └── Nodo SIEM (peer independiente)
        ├── Nivel 0: archivo exportado manualmente
        ├── Nivel 1: API polling (lectura)
        └── Nivel 2: webhook / stream tiempo real
```

## Bloques del notebook
1. Identidad y jerarquía — `org_config.json` + nodos
2. Embudo del principal — fuentes internas del negocio
3. Embudo del nodo SIEM — conector especializado multi-formato
4. Construcción del manifold geométrico — qué es normal
5. Score de disonancia y reporte bimestral

## Lo que NO hace este notebook
- No ejecuta acciones sobre sistemas del cliente
- No genera alertas automáticas
- No reemplaza el SIEM del cliente
- No requiere credenciales de escritura sobre ningún sistema

---
> **Formatos SIEM soportados:** Chronicle UDM · Splunk JSON · Microsoft Sentinel · CEF genérico · CSV sin estructura

## Instalación de dependencias

In [ ]:
!pip install gradio numpy pandas matplotlib plotly requests python-dateutil rich -q
print('✅ Dependencias listas.')

## Bloque 1 — Identidad y jerarquía

Carga el `org_config.json` del principal, registra los nodos vinculados
y establece qué nodo es el SIEM y en qué nivel de despliegue está.

### Estructura mínima de `org_config.json`
```json
{
  "org_name": "Mi Empresa SA",
  "org_type": "legal | healthcare | industrial | hotel | generic",
  "principal_id": "ORG-001",
  "users": [{"id": "u01", "name": "Ana", "role": "admin",
              "typical_ips": ["192.168.1.10"], "typical_hours": [8, 20]}],
  "assets": [{"id": "a01", "name": "Sistema ERP", "sensitivity": "high"}],
  "nodes": [
    {"node_id": "SIEM-001", "node_type": "siem",
     "siem_platform": "chronicle",
     "deployment_level": 0,
     "api_endpoint": "",
     "api_key": "",
     "webhook_url": ""}
  ],
  "thresholds": {
    "geo_dissonance_alert": 0.25,
    "geo_dissonance_report": 0.40
  },
  "bimester": "2025-B1"
}
```

In [ ]:
import json, os, uuid, hashlib, copy
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

# ─── Config por defecto (despacho jurídico) ───────────────────
DEFAULT_CONFIG = {
    "org_name":     "Despacho Jurídico — Demo GSL",
    "org_type":     "legal",
    "principal_id": "ORG-DEMO-001",
    "bimester":     "2025-B1",
    "users": [
        {"id": "u01", "name": "Ana Socia (admin)",     "role": "admin",
         "typical_ips": ["192.168.1.10"],              "typical_hours": [8, 20]},
        {"id": "u02", "name": "Luis Colaborador Ext.", "role": "external",
         "typical_ips": ["10.0.0.5"],                 "typical_hours": [9, 18]},
        {"id": "u03", "name": "María Asociada",        "role": "staff",
         "typical_ips": ["192.168.1.12"],             "typical_hours": [9, 19]},
        {"id": "u04", "name": "Carlos Paralegal",      "role": "staff",
         "typical_ips": ["192.168.1.13"],             "typical_hours": [9, 18]}
    ],
    "assets": [
        {"id": "a01", "name": "Expediente Caso Alpha",  "sensitivity": "high"},
        {"id": "a02", "name": "Expediente Caso Beta",   "sensitivity": "high"},
        {"id": "a03", "name": "Contratos con Clientes", "sensitivity": "high"},
        {"id": "a04", "name": "Biblioteca Plantillas",  "sensitivity": "medium"},
        {"id": "a05", "name": "Agenda y Citas",         "sensitivity": "low"}
    ],
    "nodes": [
        {
            "node_id":          "SIEM-001",
            "node_type":        "siem",
            "siem_platform":    "chronicle",
            "deployment_level": 0,
            "api_endpoint":     "",
            "api_key":          "",
            "webhook_url":      "",
            "poll_interval_min": 30
        }
    ],
    "thresholds": {
        "geo_dissonance_alert":  0.25,
        "geo_dissonance_report": 0.40
    }
}

# ─── Carga y merge con config del cliente ─────────────────────
def load_config(path: str = 'org_config.json') -> dict:
    p = Path(path)
    if p.exists():
        with open(p) as f:
            custom = json.load(f)
        merged = {**DEFAULT_CONFIG, **custom}
        for key in ('thresholds',):
            if key in DEFAULT_CONFIG and key in custom:
                merged[key] = {**DEFAULT_CONFIG[key], **custom[key]}
        print(f'✅ Config cargada: {merged["org_name"]}')
    else:
        merged = DEFAULT_CONFIG.copy()
        print(f'ℹ️  Sin org_config.json — usando demo: {merged["org_name"]}')
    return merged

CFG = load_config()

# ─── Registro de jerarquía ────────────────────────────────────
RUN_ID    = f"GSL-M1-{CFG['principal_id']}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
BIMESTER  = CFG.get('bimester', '2025-B1')
ARTIFACTS = Path('gsl_artifacts') / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=True)

SIEM_NODES = [n for n in CFG.get('nodes', []) if n.get('node_type') == 'siem']
OTHER_NODES = [n for n in CFG.get('nodes', []) if n.get('node_type') != 'siem']

print(f'\n  Run ID     : {RUN_ID}')
print(f'  Bimestre   : {BIMESTER}')
print(f'  Principal  : {CFG["org_name"]}')
print(f'  Usuarios   : {len(CFG["users"])}')
print(f'  Activos    : {len(CFG["assets"])}')
print(f'  Nodos SIEM : {len(SIEM_NODES)}')
for n in SIEM_NODES:
    lvl = n.get('deployment_level', 0)
    lvl_label = {0: 'Archivo local', 1: 'API polling', 2: 'Stream/webhook'}[lvl]
    print(f'    [{n["node_id"]}] plataforma={n["siem_platform"]} nivel={lvl} ({lvl_label})')
print(f'  Artefactos : {ARTIFACTS}')

## Bloque 2 — Embudo del principal

Ingesta las fuentes internas del negocio usando el patrón del Módulo Maestro.
Neutral — no interpreta dominio. Produce el `NormalizedBundle` del principal.

**En producción:** reemplaza los mocks por lecturas reales de CSV / DB / API.
La interfaz del conector es la misma — el resto del pipeline no cambia.

In [ ]:
# ─── Generador de datos sintéticos del principal ─────────────
# En producción: sustituir por lecturas reales

import random
from dateutil.relativedelta import relativedelta

def generate_principal_events(cfg: dict, n_events: int = 300,
                               seed: int = 42) -> pd.DataFrame:
    """Genera log sintético de eventos del sistema principal."""
    rng   = random.Random(seed)
    users = cfg['users']
    assets = cfg['assets']
    actions = ['read', 'write', 'export', 'login', 'modify', 'delete', 'share']
    action_weights = [40, 20, 10, 15, 8, 3, 4]

    base = datetime(2025, 1, 1, 9, 0, 0)
    rows = []
    for i in range(n_events):
        u = rng.choice(users)
        a = rng.choice(assets)
        action = rng.choices(actions, weights=action_weights, k=1)[0]
        h_start, h_end = u['typical_hours']
        # 90% en horario normal, 10% fuera
        if rng.random() < 0.90:
            hour = rng.randint(h_start, h_end)
        else:
            hour = rng.choice([rng.randint(0, h_start-1), rng.randint(h_end+1, 23)])
        day_offset = rng.randint(0, 59)  # 2 meses
        ts = base.replace(hour=hour, minute=rng.randint(0,59)) \
                 + __import__('datetime').timedelta(days=day_offset)
        # IP: 95% típica, 5% desconocida
        ip = rng.choice(u['typical_ips']) if rng.random() < 0.95 \
             else f'203.{rng.randint(1,254)}.{rng.randint(1,254)}.{rng.randint(1,254)}'
        rows.append({
            'timestamp':    ts.isoformat(),
            'user_id':      u['id'],
            'user_name':    u['name'],
            'role':         u['role'],
            'source_ip':    ip,
            'action':       action,
            'asset_id':     a['id'],
            'asset_name':   a['name'],
            'sensitivity':  a['sensitivity'],
            'success':      rng.random() > 0.05,
            'session_id':   f'SES-{u["id"]}-{day_offset:03d}',
        })
    return pd.DataFrame(rows)


# ─── Normalización universal (igual que Módulo Maestro) ───────
def normalize_principal_bundle(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df['hour']      = df['timestamp'].dt.hour
    df['weekday']   = df['timestamp'].dt.weekday
    df['row_id']    = [str(uuid.uuid4())[:8] for _ in range(len(df))]
    df = df.dropna(subset=['timestamp'])
    return df


print('⚙️  Generando bundle del principal...')
principal_raw  = generate_principal_events(CFG)
principal_norm = normalize_principal_bundle(principal_raw)

# Persistir artefacto
principal_norm.to_csv(ARTIFACTS / 'principal_normalized_bundle.csv', index=False)

print(f'✅ Principal NormalizedBundle: {len(principal_norm)} eventos')
print(f'   Rango temporal : {principal_norm["timestamp"].min()} → {principal_norm["timestamp"].max()}')
print(f'   Usuarios únicos: {principal_norm["user_id"].nunique()}')
print(f'   Activos únicos : {principal_norm["asset_id"].nunique()}')
principal_norm.head(3)

## Bloque 3 — Embudo del nodo SIEM

Conector especializado con tres niveles de despliegue.
Detección automática de formato: Chronicle UDM, Splunk JSON, Sentinel, CEF, CSV genérico.

**Nivel 0** — archivo exportado manualmente (default, cero fricción)
**Nivel 1** — API polling cada N minutos (requiere API key de solo lectura)
**Nivel 2** — webhook / stream tiempo real (requiere regla de forwarding en el SIEM)

In [ ]:
# ─── Detectores de formato SIEM ───────────────────────────────

class SIEMFormat:
    CHRONICLE = 'chronicle_udm'
    SPLUNK    = 'splunk_json'
    SENTINEL  = 'sentinel_json'
    CEF       = 'cef_generic'
    CSV       = 'csv_generic'
    UNKNOWN   = 'unknown'


def detect_siem_format(data: Any) -> str:
    """
    Detecta el formato de un reporte SIEM.
    Acepta: dict (JSON parseado), str (JSON/CEF raw), pd.DataFrame (CSV).
    """
    if isinstance(data, pd.DataFrame):
        cols = set(data.columns.str.lower())
        if {'metadata.event_type', 'principal.user.userid'} & cols:
            return SIEMFormat.CHRONICLE
        if {'_time', 'sourcetype', 'host'} & cols:
            return SIEMFormat.SPLUNK
        if {'timegenerated', 'operationname', 'category'} & cols:
            return SIEMFormat.SENTINEL
        return SIEMFormat.CSV

    if isinstance(data, str):
        if data.strip().startswith('CEF:'):
            return SIEMFormat.CEF
        try:
            parsed = json.loads(data)
            return detect_siem_format(parsed)
        except:
            return SIEMFormat.CSV

    if isinstance(data, dict):
        keys = set(data.keys())
        if 'metadata' in keys and 'principal' in keys:
            return SIEMFormat.CHRONICLE
        if 'result' in keys and 'sourcetype' in str(data):
            return SIEMFormat.SPLUNK
        if 'TimeGenerated' in keys or 'OperationName' in keys:
            return SIEMFormat.SENTINEL

    if isinstance(data, list) and data:
        return detect_siem_format(data[0])

    return SIEMFormat.UNKNOWN


# ─── Parsers por formato ──────────────────────────────────────
# Esquema mínimo común de salida:
# timestamp | source_entity | target_entity | event_type |
# severity  | rule_id       | outcome       | raw_platform

def parse_chronicle_udm(events: list) -> pd.DataFrame:
    rows = []
    for e in events:
        meta   = e.get('metadata', {})
        prin   = e.get('principal', {})
        target = e.get('target', {})
        sec    = e.get('security_result', [{}])
        sec    = sec[0] if isinstance(sec, list) and sec else sec
        rows.append({
            'timestamp':     meta.get('eventTimestamp', meta.get('event_timestamp', '')),
            'source_entity': prin.get('user', {}).get('userid',
                             prin.get('hostname', prin.get('ip', 'unknown'))),
            'target_entity': target.get('asset', {}).get('assetId',
                             target.get('hostname', target.get('url', 'unknown'))),
            'event_type':    meta.get('eventType', meta.get('event_type', 'UNKNOWN')),
            'severity':      sec.get('severity', 'UNKNOWN') if isinstance(sec, dict) else 'UNKNOWN',
            'rule_id':       sec.get('ruleName', '') if isinstance(sec, dict) else '',
            'outcome':       sec.get('action', ['UNKNOWN'])[0] if isinstance(sec, dict)
                             and isinstance(sec.get('action'), list) else 'UNKNOWN',
            'raw_platform':  SIEMFormat.CHRONICLE,
        })
    return pd.DataFrame(rows)


def parse_splunk_json(events: list) -> pd.DataFrame:
    rows = []
    for e in events:
        result = e.get('result', e)  # Splunk puede envolver en 'result'
        rows.append({
            'timestamp':     result.get('_time', result.get('timestamp', '')),
            'source_entity': result.get('user', result.get('src', result.get('src_ip', 'unknown'))),
            'target_entity': result.get('dest', result.get('dest_ip', result.get('host', 'unknown'))),
            'event_type':    result.get('EventCode', result.get('event_type', result.get('sourcetype', 'UNKNOWN'))),
            'severity':      result.get('severity', result.get('urgency', 'UNKNOWN')),
            'rule_id':       result.get('rule_name', result.get('search_name', '')),
            'outcome':       result.get('action', result.get('status', 'UNKNOWN')),
            'raw_platform':  SIEMFormat.SPLUNK,
        })
    return pd.DataFrame(rows)


def parse_sentinel_json(events: list) -> pd.DataFrame:
    rows = []
    for e in events:
        rows.append({
            'timestamp':     e.get('TimeGenerated', e.get('timegenerated', '')),
            'source_entity': e.get('Account', e.get('UserPrincipalName',
                             e.get('CallerIpAddress', 'unknown'))),
            'target_entity': e.get('ResourceId', e.get('OperationName', 'unknown')),
            'event_type':    e.get('OperationName', e.get('Category', 'UNKNOWN')),
            'severity':      e.get('Level', e.get('AlertSeverity', 'UNKNOWN')),
            'rule_id':       e.get('CorrelationId', e.get('AlertName', '')),
            'outcome':       e.get('ResultType', e.get('ActivityStatus', 'UNKNOWN')),
            'raw_platform':  SIEMFormat.SENTINEL,
        })
    return pd.DataFrame(rows)


def parse_cef(raw: str) -> pd.DataFrame:
    """Parser CEF (Common Event Format) genérico."""
    rows = []
    for line in raw.strip().split('\n'):
        if not line.startswith('CEF:'): continue
        parts = line.split('|')
        if len(parts) < 8: continue
        ext_str = parts[7] if len(parts) > 7 else ''
        ext = {}
        for kv in ext_str.split():
            if '=' in kv:
                k, v = kv.split('=', 1)
                ext[k] = v
        rows.append({
            'timestamp':     ext.get('rt', ext.get('start', '')),
            'source_entity': ext.get('suser', ext.get('src', parts[4] if len(parts)>4 else 'unknown')),
            'target_entity': ext.get('dhost', ext.get('dst', ext.get('destinationHostName', 'unknown'))),
            'event_type':    parts[5] if len(parts) > 5 else 'UNKNOWN',
            'severity':      parts[6] if len(parts) > 6 else 'UNKNOWN',
            'rule_id':       parts[4] if len(parts) > 4 else '',
            'outcome':       ext.get('act', 'UNKNOWN'),
            'raw_platform':  SIEMFormat.CEF,
        })
    return pd.DataFrame(rows)


def parse_csv_generic(df: pd.DataFrame) -> pd.DataFrame:
    """Parser CSV genérico — mapea las columnas más comunes."""
    col_map = {
        'timestamp':     ['timestamp', 'time', 'datetime', 'date', '_time', 'timegenerated'],
        'source_entity': ['user', 'username', 'src_user', 'source_user', 'account', 'userid'],
        'target_entity': ['target', 'dest', 'destination', 'resource', 'host', 'asset'],
        'event_type':    ['event_type', 'eventtype', 'action', 'category', 'operationname'],
        'severity':      ['severity', 'level', 'priority', 'urgency', 'alertseverity'],
        'rule_id':       ['rule', 'rule_name', 'alert_name', 'detection', 'ruleid'],
        'outcome':       ['outcome', 'result', 'status', 'resulttype', 'activitystatus'],
    }
    cols_lower = {c.lower(): c for c in df.columns}
    out = pd.DataFrame()
    for field, candidates in col_map.items():
        for cand in candidates:
            if cand in cols_lower:
                out[field] = df[cols_lower[cand]].astype(str)
                break
        if field not in out.columns:
            out[field] = 'unknown'
    out['raw_platform'] = SIEMFormat.CSV
    return out


def parse_siem_data(data: Any, fmt: str = None) -> pd.DataFrame:
    """Router principal — detecta formato y aplica el parser correcto."""
    if fmt is None:
        fmt = detect_siem_format(data)
    print(f'   Formato detectado: {fmt}')

    if fmt == SIEMFormat.CHRONICLE:
        events = data if isinstance(data, list) else [data]
        return parse_chronicle_udm(events)
    elif fmt == SIEMFormat.SPLUNK:
        events = data if isinstance(data, list) else [data]
        return parse_splunk_json(events)
    elif fmt == SIEMFormat.SENTINEL:
        events = data if isinstance(data, list) else [data]
        return parse_sentinel_json(events)
    elif fmt == SIEMFormat.CEF:
        return parse_cef(data if isinstance(data, str) else json.dumps(data))
    elif fmt == SIEMFormat.CSV:
        df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        return parse_csv_generic(df)
    else:
        print('   ⚠️  Formato desconocido — tratando como CSV genérico')
        df = data if isinstance(data, pd.DataFrame) else pd.DataFrame([data])
        return parse_csv_generic(df)


print('✅ Parsers SIEM listos: Chronicle UDM · Splunk · Sentinel · CEF · CSV')

In [ ]:
# ─── Conector SIEM multi-nivel ────────────────────────────────

def generate_mock_siem_events(platform: str = 'chronicle',
                               n: int = 150, seed: int = 7) -> Any:
    """Genera eventos SIEM sintéticos por plataforma para pruebas."""
    rng = random.Random(seed)
    base = datetime(2025, 1, 1, tzinfo=timezone.utc)

    event_types = ['USER_LOGIN', 'FILE_OPEN', 'FILE_COPY', 'NETWORK_CONNECTION',
                   'PROCESS_LAUNCH', 'USER_LOGOUT', 'PERMISSION_CHANGE']
    severities  = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
    sev_weights = [50, 30, 15, 5]
    users_sample = [u['id'] for u in CFG['users']]

    if platform == 'chronicle':
        events = []
        for i in range(n):
            ts = base + __import__('datetime').timedelta(
                days=rng.randint(0,59), hours=rng.randint(0,23),
                minutes=rng.randint(0,59))
            sev = rng.choices(severities, weights=sev_weights, k=1)[0]
            events.append({
                'metadata': {
                    'eventTimestamp': ts.isoformat(),
                    'eventType': rng.choice(event_types),
                    'id': str(uuid.uuid4())
                },
                'principal': {
                    'user': {'userid': rng.choice(users_sample)},
                    'ip': [f'192.168.{rng.randint(1,5)}.{rng.randint(1,200)}']
                },
                'target': {
                    'asset': {'assetId': rng.choice([a['id'] for a in CFG['assets']])},
                    'url': f'https://internal.corp/{rng.choice(["docs","erp","mail"])}/'
                },
                'security_result': [{
                    'severity': sev,
                    'ruleName': f'RULE-{rng.randint(100,999)}',
                    'action': [rng.choice(['ALLOW','BLOCK','UNKNOWN'])]
                }]
            })
        return events

    elif platform == 'splunk':
        events = []
        for i in range(n):
            ts = base + __import__('datetime').timedelta(
                days=rng.randint(0,59), hours=rng.randint(0,23))
            events.append({'result': {
                '_time':      ts.strftime('%Y-%m-%dT%H:%M:%S'),
                'user':       rng.choice(users_sample),
                'src':        f'10.0.{rng.randint(0,5)}.{rng.randint(1,200)}',
                'dest':       f'srv-{rng.randint(1,10):02d}.internal',
                'EventCode':  str(rng.choice([4624,4625,4648,4720,4732])),
                'severity':   rng.choices(severities, weights=sev_weights, k=1)[0],
                'rule_name':  f'Detection-{rng.randint(10,99)}',
                'action':     rng.choice(['allowed','blocked','unknown']),
                'sourcetype': rng.choice(['WinEventLog','syslog','auth'])
            }})
        return events

    elif platform == 'sentinel':
        events = []
        for i in range(n):
            ts = base + __import__('datetime').timedelta(
                days=rng.randint(0,59), hours=rng.randint(0,23))
            events.append({
                'TimeGenerated':   ts.isoformat() + 'Z',
                'Account':         rng.choice(users_sample),
                'CallerIpAddress': f'172.16.{rng.randint(0,5)}.{rng.randint(1,200)}',
                'OperationName':   rng.choice(['SignIn','FileAccess','PolicyChange']),
                'ResourceId':      f'/subscriptions/xxx/resources/res-{rng.randint(1,10)}',
                'Level':           rng.choices(['Information','Warning','Error','Critical'],
                                              weights=[50,30,15,5],k=1)[0],
                'CorrelationId':   str(uuid.uuid4()),
                'ResultType':      rng.choice(['Success','Failure','Timeout']),
                'Category':        rng.choice(['AuditLogs','SignInLogs','SecurityEvent'])
            })
        return events

    else:  # CEF
        lines = []
        for i in range(n):
            ts_ms = int((base + __import__('datetime').timedelta(
                days=rng.randint(0,59), hours=rng.randint(0,23))).timestamp() * 1000)
            sev_num = rng.choices([1,3,7,10], weights=sev_weights, k=1)[0]
            usr = rng.choice(users_sample)
            dst = f'srv-{rng.randint(1,10):02d}.internal'
            act = rng.choice(['permit','deny','drop'])
            lines.append(
                f'CEF:0|VendorX|ProductY|1.0|{rng.randint(100,999)}|'
                f'{rng.choice(event_types)}|{sev_num}|'
                f'rt={ts_ms} suser={usr} dhost={dst} act={act}'
            )
        return '\n'.join(lines)


def ingest_siem_node(node_cfg: dict, mock: bool = True) -> pd.DataFrame:
    """
    Ingesta datos del nodo SIEM según su nivel de despliegue.
    mock=True usa datos sintéticos. En producción: mock=False.
    """
    platform = node_cfg.get('siem_platform', 'chronicle')
    level    = node_cfg.get('deployment_level', 0)
    node_id  = node_cfg.get('node_id', 'SIEM-001')

    print(f'\n📡 Nodo SIEM: {node_id} | Plataforma: {platform} | Nivel: {level}')

    if mock:
        print('   Modo: datos sintéticos (mock=True)')
        raw_data = generate_mock_siem_events(platform)

    elif level == 0:
        # ── Nivel 0: archivo local ──────────────────────────────
        siem_file = node_cfg.get('siem_file_path', f'siem_export_{platform}.json')
        print(f'   Nivel 0 — leyendo archivo: {siem_file}')
        p = Path(siem_file)
        if not p.exists():
            raise FileNotFoundError(
                f'Archivo SIEM no encontrado: {siem_file}\n'
                f'Exporta un reporte de {platform} y colócalo en: {p.absolute()}')
        if p.suffix == '.json':
            with open(p) as f:
                raw_data = json.load(f)
        elif p.suffix == '.csv':
            raw_data = pd.read_csv(p)
        else:
            with open(p) as f:
                raw_data = f.read()

    elif level == 1:
        # ── Nivel 1: API polling ────────────────────────────────
        import requests
        endpoint = node_cfg.get('api_endpoint', '')
        api_key  = node_cfg.get('api_key', '')
        if not endpoint or not api_key:
            raise ValueError('Nivel 1 requiere api_endpoint y api_key en org_config.json')
        print(f'   Nivel 1 — polling API: {endpoint}')
        headers = {'Authorization': f'Bearer {api_key}',
                   'Content-Type': 'application/json'}
        # Endpoint estándar de Chronicle DataAccessService
        # Adaptar según la plataforma del cliente
        params  = {'startTime': (datetime.now(timezone.utc) -
                   __import__('datetime').timedelta(
                       minutes=node_cfg.get('poll_interval_min', 30))).isoformat(),
                   'endTime': datetime.now(timezone.utc).isoformat()}
        resp = requests.get(endpoint, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        raw_data = resp.json()

    elif level == 2:
        # ── Nivel 2: stream / webhook ───────────────────────────
        # En este modo el notebook actúa como receptor.
        # El stream se maneja externamente; aquí procesamos el buffer acumulado.
        buffer_path = node_cfg.get('stream_buffer_path', f'siem_stream_buffer_{node_id}.jsonl')
        print(f'   Nivel 2 — leyendo buffer de stream: {buffer_path}')
        p = Path(buffer_path)
        if not p.exists():
            raise FileNotFoundError(f'Buffer de stream no encontrado: {buffer_path}')
        with open(p) as f:
            raw_data = [json.loads(line) for line in f if line.strip()]

    else:
        raise ValueError(f'Nivel de despliegue inválido: {level}. Use 0, 1 o 2.')

    # ── Parsear al esquema mínimo común ──────────────────────
    df = parse_siem_data(raw_data)

    # ── Normalización universal ───────────────────────────────
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df['hour']      = df['timestamp'].dt.hour
    df['weekday']   = df['timestamp'].dt.weekday
    df['node_id']   = node_id
    df['row_id']    = [str(uuid.uuid4())[:8] for _ in range(len(df))]
    df = df.dropna(subset=['timestamp'])

    print(f'   ✅ {len(df)} eventos normalizados')
    print(f'   Severidades: {df["severity"].value_counts().to_dict()}')
    print(f'   Tipos de evento: {df["event_type"].nunique()} únicos')

    return df


# ─── Ejecutar para todos los nodos SIEM registrados ──────────
siem_bundles = {}
for node in SIEM_NODES:
    df_siem = ingest_siem_node(node, mock=True)  # mock=False en producción
    siem_bundles[node['node_id']] = df_siem
    df_siem.to_csv(ARTIFACTS / f'siem_{node["node_id"]}_normalized_bundle.csv', index=False)

if not SIEM_NODES:
    print('ℹ️  Sin nodos SIEM configurados — solo se procesará el bundle del principal.')

## Bloque 4 — Construcción del manifold geométrico

Procesa el bundle del principal y los bundles de nodos SIEM por separado.
Construye un **vector de firma** por cada extremo (usuario / entidad).

Este bloque **solo aprende** qué es normal. No detecta ataques todavía.
Es el período de observación del Modo 1 — el equivalente al Modo 1 pasivo
que el cliente valida antes de activar el Modo 2.

### Dimensiones del vector de firma (8D)
```
[0] ip_consistency      — fracción de accesos desde IPs conocidas
[1] hour_centrality     — qué tan centrado está el horario de actividad
[2] action_entropy      — diversidad de tipos de acción
[3] sensitivity_bias    — fracción de accesos a activos de alta sensibilidad
[4] session_regularity  — varianza de duración/cadencia de sesiones
[5] failure_rate        — fracción de acciones fallidas
[6] event_velocity      — eventos por hora promedio
[7] cross_asset_spread  — cuántos activos distintos toca por sesión
```

In [ ]:
# ─── Constructor de vectores de firma ────────────────────────

def build_signature_vector(df: pd.DataFrame, entity_id: str,
                            entity_col: str, cfg: dict) -> np.ndarray:
    """
    Construye el vector de firma 8D de un extremo (usuario o entidad SIEM)
    a partir de su historial de eventos en el bundle.
    """
    subset = df[df[entity_col] == entity_id]
    if len(subset) < 3:
        return np.full(8, 0.5)  # vector neutro por falta de datos

    # [0] ip_consistency
    user_cfg = next((u for u in cfg.get('users', [])
                     if u['id'] == entity_id), None)
    if user_cfg and 'source_ip' in subset.columns:
        known_ips = set(user_cfg.get('typical_ips', []))
        ip_cons = subset['source_ip'].isin(known_ips).mean() if known_ips else 0.5
    elif 'source_ip' in subset.columns:
        # Para entidades SIEM: consistencia = 1 - fracción de IPs únicas relativas
        ip_cons = 1 - (subset['source_ip'].nunique() / max(len(subset), 1))
    else:
        ip_cons = 0.5

    # [1] hour_centrality — qué tan predecible es el horario
    if 'hour' in subset.columns:
        hours = subset['hour'].dropna().values
        if len(hours) > 1:
            h_std = np.std(hours)
            hour_cent = 1 - min(h_std / 12.0, 1.0)
        else:
            hour_cent = 0.5
    else:
        hour_cent = 0.5

    # [2] action_entropy — diversidad de acciones
    action_col = 'action' if 'action' in subset.columns else 'event_type'
    if action_col in subset.columns:
        counts = subset[action_col].value_counts(normalize=True)
        entropy = float(-np.sum(counts * np.log2(counts + 1e-10)))
        max_entropy = np.log2(max(len(counts), 2))
        act_ent = entropy / max_entropy
    else:
        act_ent = 0.5

    # [3] sensitivity_bias — accesos a activos sensibles
    if 'sensitivity' in subset.columns:
        sens_bias = (subset['sensitivity'] == 'high').mean()
    else:
        # Para SIEM: fracción de eventos HIGH/CRITICAL
        if 'severity' in subset.columns:
            sens_bias = subset['severity'].isin(['HIGH','CRITICAL','high','critical']).mean()
        else:
            sens_bias = 0.3

    # [4] session_regularity
    sess_col = 'session_id' if 'session_id' in subset.columns else None
    if sess_col:
        sess_counts = subset[sess_col].value_counts()
        sess_reg = 1 - min(np.std(sess_counts.values) / max(np.mean(sess_counts.values), 1), 1.0)
    else:
        sess_reg = 0.5

    # [5] failure_rate
    fail_col = 'success' if 'success' in subset.columns else \
               'outcome' if 'outcome' in subset.columns else None
    if fail_col == 'success':
        fail_rate = 1 - subset['success'].astype(float).mean()
    elif fail_col == 'outcome':
        fail_rate = subset['outcome'].isin(
            ['BLOCK','blocked','Failure','deny','drop','UNKNOWN']).mean()
    else:
        fail_rate = 0.05

    # [6] event_velocity — eventos por hora
    if 'timestamp' in subset.columns and len(subset) > 1:
        ts = subset['timestamp'].dropna().sort_values()
        span_hours = max((ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600, 1)
        velocity = len(subset) / span_hours
        ev_vel = min(velocity / 50.0, 1.0)  # normalizar a 50 eventos/hora = 1.0
    else:
        ev_vel = 0.3

    # [7] cross_asset_spread
    asset_col = 'asset_id' if 'asset_id' in subset.columns else \
                'target_entity' if 'target_entity' in subset.columns else None
    if asset_col and sess_col:
        spread_per_sess = subset.groupby(sess_col)[asset_col].nunique()
        spread = float(spread_per_sess.mean()) / max(len(cfg.get('assets', [1])), 1)
    elif asset_col:
        spread = subset[asset_col].nunique() / max(len(cfg.get('assets', [1])), 1)
    else:
        spread = 0.3

    vec = np.array([
        float(np.clip(ip_cons,    0, 1)),
        float(np.clip(hour_cent,  0, 1)),
        float(np.clip(act_ent,    0, 1)),
        float(np.clip(sens_bias,  0, 1)),
        float(np.clip(sess_reg,   0, 1)),
        float(np.clip(fail_rate,  0, 1)),
        float(np.clip(ev_vel,     0, 1)),
        float(np.clip(spread,     0, 1)),
    ])
    return vec


def build_manifold(df: pd.DataFrame, entity_col: str,
                   cfg: dict, label: str = '') -> dict:
    """
    Construye el manifold completo para todos los extremos en el bundle.
    Devuelve dict: entity_id → {'vector': np.array, 'n_events': int}
    """
    entities = df[entity_col].dropna().unique()
    manifold = {}
    for eid in entities:
        vec = build_signature_vector(df, eid, entity_col, cfg)
        n   = len(df[df[entity_col] == eid])
        manifold[eid] = {'vector': vec, 'n_events': n, 'entity_col': entity_col}

    print(f'✅ Manifold [{label}]: {len(manifold)} extremos')
    for eid, m in manifold.items():
        print(f'   {eid:<30} n={m["n_events"]:>4}  '
              f'vec=[{" ".join(f"{v:.2f}" for v in m["vector"])}]')
    return manifold


# ─── Construir manifolds ──────────────────────────────────────
print('⚙️  Construyendo manifold del principal...')
manifold_principal = build_manifold(
    principal_norm, entity_col='user_id', cfg=CFG, label='Principal')

manifold_siem = {}
for node_id, df_siem in siem_bundles.items():
    print(f'\n⚙️  Construyendo manifold SIEM [{node_id}]...')
    manifold_siem[node_id] = build_manifold(
        df_siem, entity_col='source_entity', cfg=CFG, label=f'SIEM-{node_id}')

# ─── Persistir manifolds ──────────────────────────────────────
def serialize_manifold(m: dict) -> dict:
    return {k: {'vector': v['vector'].tolist(),
                'n_events': v['n_events'],
                'entity_col': v['entity_col']} for k, v in m.items()}

with open(ARTIFACTS / 'manifold_principal.json', 'w') as f:
    json.dump(serialize_manifold(manifold_principal), f, indent=2)
for node_id, m in manifold_siem.items():
    with open(ARTIFACTS / f'manifold_siem_{node_id}.json', 'w') as f:
        json.dump(serialize_manifold(m), f, indent=2)
print('\n✅ Manifolds persistidos en', ARTIFACTS)

## Bloque 5 — Score de disonancia y reporte bimestral

Compara el estado actual de cada extremo contra su manifold histórico.
Genera el score de disonancia por sesión, por usuario y por flujo SIEM.

**En Modo 1:** solo visibilidad. Sin alertas automáticas.
El cliente revisa el reporte y decide si escalar al Modo 2.

In [ ]:
# ─── Motor de disonancia ──────────────────────────────────────

def compute_dissonance_from_vector(
    current_vec: np.ndarray,
    baseline_vec: np.ndarray,
    weights: np.ndarray = None,
) -> float:
    """
    Disonancia geométrica: distancia ponderada en el espacio de firma.
    Normalizada a [0, 1] — 0 = idéntico al baseline, 1 = máxima divergencia.
    """
    if weights is None:
        # Pesos por dimensión — ip e ip_hour más importantes
        weights = np.array([0.25, 0.20, 0.10, 0.15, 0.08, 0.10, 0.07, 0.05])
    weights = weights / weights.sum()
    diff    = np.abs(current_vec - baseline_vec)
    return float(np.dot(diff, weights))


def score_entity_session(
    df: pd.DataFrame,
    entity_id: str,
    entity_col: str,
    manifold: dict,
    cfg: dict,
    window_hours: int = 2,
) -> List[dict]:
    """
    Calcula el score de disonancia para ventanas temporales del extremo.
    Devuelve lista de scores por ventana con metadatos.
    """
    if entity_id not in manifold:
        return []

    baseline = manifold[entity_id]['vector']
    subset   = df[df[entity_col] == entity_id].copy()
    if len(subset) < 2:
        return []

    subset = subset.sort_values('timestamp')
    ts_min = subset['timestamp'].min()
    ts_max = subset['timestamp'].max()

    results = []
    window  = __import__('datetime').timedelta(hours=window_hours)
    current = ts_min

    while current < ts_max:
        end = current + window
        win = subset[(subset['timestamp'] >= current) & (subset['timestamp'] < end)]
        if len(win) < 2:
            current = end
            continue

        current_vec = build_signature_vector(win, entity_id, entity_col, cfg)
        dis = compute_dissonance_from_vector(current_vec, baseline)

        results.append({
            'entity_id':   entity_id,
            'window_start': current.isoformat(),
            'window_end':   end.isoformat(),
            'n_events':     len(win),
            'dissonance':   round(dis, 4),
            'current_vector': current_vec.tolist(),
            'baseline_vector': baseline.tolist(),
        })
        current = end

    return results


def score_all_entities(
    df: pd.DataFrame,
    entity_col: str,
    manifold: dict,
    cfg: dict,
    label: str = '',
) -> pd.DataFrame:
    all_scores = []
    for eid in manifold:
        scores = score_entity_session(df, eid, entity_col, manifold, cfg)
        all_scores.extend(scores)
    if not all_scores:
        return pd.DataFrame()
    df_scores = pd.DataFrame(all_scores)
    df_scores['source'] = label
    return df_scores


# ─── Calcular scores ──────────────────────────────────────────
print('⚙️  Calculando scores de disonancia...')

scores_principal = score_all_entities(
    principal_norm, 'user_id', manifold_principal, CFG, 'principal')

scores_siem_all = []
for node_id, df_siem in siem_bundles.items():
    sc = score_all_entities(
        df_siem, 'source_entity', manifold_siem[node_id], CFG, f'siem_{node_id}')
    scores_siem_all.append(sc)

scores_siem = pd.concat(scores_siem_all, ignore_index=True) \
    if scores_siem_all else pd.DataFrame()

all_scores = pd.concat(
    [s for s in [scores_principal, scores_siem] if not s.empty],
    ignore_index=True
)

th_alert  = CFG['thresholds']['geo_dissonance_alert']
th_report = CFG['thresholds']['geo_dissonance_report']

print(f'\n✅ Scores calculados: {len(all_scores)} ventanas temporales')
if not all_scores.empty:
    print(f'   Disonancia media   : {all_scores["dissonance"].mean():.3f}')
    print(f'   Disonancia máxima  : {all_scores["dissonance"].max():.3f}')
    n_alert  = (all_scores['dissonance'] >= th_alert).sum()
    n_report = (all_scores['dissonance'] >= th_report).sum()
    print(f'   Ventanas ≥ {th_alert} (alerta)  : {n_alert}')
    print(f'   Ventanas ≥ {th_report} (reporte) : {n_report}')

all_scores.to_csv(ARTIFACTS / 'dissonance_scores.csv', index=False)
print(f'   Artefacto: {ARTIFACTS}/dissonance_scores.csv')

In [ ]:
# ─── Reporte bimestral ────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'


def plot_dissonance_heatmap(scores: pd.DataFrame,
                             title: str = '') -> go.Figure:
    """Mapa de calor: entidad × tiempo con color = disonancia."""
    if scores.empty: return go.Figure()
    pivot = scores.pivot_table(
        index='entity_id',
        columns='window_start',
        values='dissonance',
        aggfunc='mean'
    ).fillna(0)
    # Truncar etiquetas de tiempo
    cols = [c[:16] for c in pivot.columns]
    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=cols,
        y=list(pivot.index),
        colorscale=[[0,'#d4edda'],[0.25,'#fff3cd'],
                    [0.50,'#FAC775'],[1.0,'#E24B4A']],
        zmin=0, zmax=0.8,
        colorbar=dict(title='Disonancia', tickformat='.2f'),
        hovertemplate='Entidad: %{y}<br>Ventana: %{x}<br>Disonancia: %{z:.3f}<extra></extra>'
    ))
    fig.update_layout(
        title=title, height=max(200, len(pivot)*40 + 100),
        margin=dict(l=120, r=40, t=60, b=80),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=11),
        xaxis_title='Ventana temporal', yaxis_title='Extremo',
    )
    return fig


def plot_dissonance_trend(scores: pd.DataFrame,
                           title: str = '') -> go.Figure:
    """Tendencia de disonancia por entidad a lo largo del bimestre."""
    if scores.empty: return go.Figure()
    fig = go.Figure()
    entities = scores['entity_id'].unique()
    palette  = ['#378ADD','#639922','#BA7517','#E24B4A','#9B59B6','#1ABC9C']
    for i, eid in enumerate(entities):
        sub = scores[scores['entity_id'] == eid].sort_values('window_start')
        fig.add_trace(go.Scatter(
            x=sub['window_start'], y=sub['dissonance'],
            mode='lines+markers', name=str(eid),
            line=dict(color=palette[i % len(palette)], width=1.5),
            marker=dict(size=5),
            hovertemplate=f'<b>{eid}</b><br>%{{x}}<br>Disonancia: %{{y:.3f}}<extra></extra>'
        ))
    fig.add_hline(y=th_alert,  line_dash='dash', line_color='#BA7517',
                  line_width=1, annotation_text=f'umbral alerta {th_alert}',
                  annotation_font_size=9)
    fig.add_hline(y=th_report, line_dash='dash', line_color='#E24B4A',
                  line_width=1, annotation_text=f'umbral reporte {th_report}',
                  annotation_font_size=9)
    fig.update_layout(
        title=title, height=350,
        margin=dict(l=40, r=120, t=60, b=60),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif', size=11),
        yaxis_range=[0, 1.05], yaxis_gridcolor='#e8e8e4',
        xaxis_title='Ventana', yaxis_title='Disonancia',
        legend=dict(orientation='v', x=1.02)
    )
    return fig


def generate_bimester_report(scores: pd.DataFrame, cfg: dict,
                              bimestre: str, run_id: str) -> str:
    """Genera el reporte bimestral en Markdown."""
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
    th_a = cfg['thresholds']['geo_dissonance_alert']
    th_r = cfg['thresholds']['geo_dissonance_report']

    if scores.empty:
        return f'# Reporte Bimestral GSL\n**Sin datos suficientes para {bimestre}.**'

    n_alert  = (scores['dissonance'] >= th_a).sum()
    n_report = (scores['dissonance'] >= th_r).sum()
    top5     = scores.nlargest(5, 'dissonance')[['entity_id','window_start','dissonance','source']]

    lines = [
        f'# 🛡️ Reporte Bimestral GSL Modo 1',
        f'**Organización:** {cfg["org_name"]}  ',
        f'**Bimestre:** {bimestre}  ',
        f'**Run ID:** {run_id}  ',
        f'**Generado:** {now}  ',
        '',
        '## Resumen ejecutivo',
        f'Se analizaron **{len(scores)}** ventanas temporales en el bimestre. '
        f'**{n_alert}** ventanas superaron el umbral de alerta ({th_a}). '
        f'**{n_report}** ventanas superaron el umbral de reporte ({th_r}).',
        '',
        '> Este reporte es **solo de observación (Modo 1)**. '
        'Ninguna acción automática fue ejecutada.',
        '',
        '## Métricas globales',
        f'| Métrica | Valor |',
        f'|---|---|',
        f'| Disonancia media | {scores["dissonance"].mean():.3f} |',
        f'| Disonancia máxima | {scores["dissonance"].max():.3f} |',
        f'| Ventanas sobre umbral alerta ({th_a}) | {n_alert} |',
        f'| Ventanas sobre umbral reporte ({th_r}) | {n_report} |',
        f'| Extremos monitoreados | {scores["entity_id"].nunique()} |',
        '',
        '## Top 5 ventanas de mayor disonancia',
        '| Extremo | Ventana | Disonancia | Fuente |',
        '|---|---|---|---|',
    ]
    for _, row in top5.iterrows():
        lines.append(f'| {row["entity_id"]} | {str(row["window_start"])[:16]} '
                     f'| **{row["dissonance"]:.3f}** | {row["source"]} |')

    lines += [
        '',
        '## Disonancia por extremo (promedio bimestral)',
        '| Extremo | Media | Máxima | Ventanas | Fuente |',
        '|---|---|---|---|---|',
    ]
    for eid, grp in scores.groupby('entity_id'):
        flag = ' ⚠️' if grp['dissonance'].max() >= th_a else ''
        src  = grp['source'].iloc[0]
        lines.append(f'| {eid}{flag} | {grp["dissonance"].mean():.3f} '
                     f'| {grp["dissonance"].max():.3f} | {len(grp)} | {src} |')

    lines += [
        '',
        '## Dimensiones del vector de firma',
        '| # | Dimensión | Descripción |',
        '|---|---|---|',
        '| 0 | ip_consistency | Fracción de accesos desde IPs conocidas |',
        '| 1 | hour_centrality | Predictibilidad del horario de actividad |',
        '| 2 | action_entropy | Diversidad de tipos de acción |',
        '| 3 | sensitivity_bias | Fracción de accesos a activos sensibles |',
        '| 4 | session_regularity | Regularidad en la cadencia de sesiones |',
        '| 5 | failure_rate | Fracción de acciones fallidas |',
        '| 6 | event_velocity | Eventos por hora promedio |',
        '| 7 | cross_asset_spread | Activos distintos tocados por sesión |',
        '',
        '## Próximos pasos sugeridos',
        f'1. Revisar los {n_report} extremos con disonancia ≥ {th_r} con el equipo.',
        '2. Validar si las anomalías detectadas corresponden a comportamiento legítimo nuevo.',
        '3. Si la precisión es satisfactoria, considerar activar **Modo 2** para los extremos validados.',
        '4. Ajustar umbrales en `org_config.json` si hay exceso de falsos positivos.',
    ]
    return '\n'.join(lines)


# ─── Ejecutar reporte ─────────────────────────────────────────
from IPython.display import display, Markdown

if not all_scores.empty:
    fig_hm = plot_dissonance_heatmap(
        all_scores, f'Mapa de disonancia — {CFG["org_name"]} — {BIMESTER}')
    fig_hm.show()

    fig_tr = plot_dissonance_trend(
        all_scores, f'Tendencia de disonancia bimestral — {BIMESTER}')
    fig_tr.show()

report_md = generate_bimester_report(all_scores, CFG, BIMESTER, RUN_ID)
display(Markdown(report_md))

with open(ARTIFACTS / f'bimester_report_{BIMESTER}.md', 'w') as f:
    f.write(report_md)
print(f'\n✅ Reporte guardado: {ARTIFACTS}/bimester_report_{BIMESTER}.md')

## Dashboard Gradio — Modo 1

Interfaz interactiva para explorar los scores bimestrales.
El cliente puede navegar por extremo, ajustar el período de análisis
y descargar el reporte.

In [ ]:
import gradio as gr

_state = {
    'scores':    all_scores,
    'manifold':  manifold_principal,
    'cfg':       CFG,
    'bimester':  BIMESTER,
    'run_id':    RUN_ID,
}


def refresh_analysis(config_json: str, siem_file: str,
                     siem_platform: str, seed: int):
    """Re-ejecuta el pipeline completo con nueva config o archivo SIEM."""
    try:
        cfg = json.loads(config_json) if config_json.strip() else CFG
        merged = {**DEFAULT_CONFIG, **cfg}
        for k in ('thresholds',):
            if k in DEFAULT_CONFIG and k in cfg:
                merged[k] = {**DEFAULT_CONFIG[k], **cfg[k]}
    except Exception as e:
        return None, None, f'❌ Config inválida: {e}', '', '', ''

    # Principal
    p_raw  = generate_principal_events(merged, seed=int(seed))
    p_norm = normalize_principal_bundle(p_raw)
    m_prin = build_manifold(p_norm, 'user_id', merged, 'Principal')
    s_prin = score_all_entities(p_norm, 'user_id', m_prin, merged, 'principal')

    # SIEM (mock)
    siem_nodes = [n for n in merged.get('nodes', []) if n.get('node_type') == 'siem']
    siem_scores_list = []
    for node in siem_nodes:
        node_copy = dict(node)
        if siem_platform:
            node_copy['siem_platform'] = siem_platform
        df_s = ingest_siem_node(node_copy, mock=True)
        m_s  = build_manifold(df_s, 'source_entity', merged, node['node_id'])
        s_s  = score_all_entities(df_s, 'source_entity', m_s, merged, f'siem_{node["node_id"]}')
        siem_scores_list.append(s_s)

    all_sc = pd.concat(
        [s for s in [s_prin] + siem_scores_list if not s.empty],
        ignore_index=True
    )
    _state.update({'scores': all_sc, 'manifold': m_prin, 'cfg': merged})

    fig_hm = plot_dissonance_heatmap(all_sc, 'Mapa de disonancia')
    fig_tr = plot_dissonance_trend(all_sc, 'Tendencia bimestral')
    report = generate_bimester_report(all_sc, merged, BIMESTER, RUN_ID)

    th_a = merged['thresholds']['geo_dissonance_alert']
    m_mean  = f"{all_sc['dissonance'].mean():.3f}" if not all_sc.empty else 'N/A'
    m_max   = f"{all_sc['dissonance'].max():.3f}" if not all_sc.empty else 'N/A'
    m_alert = str((all_sc['dissonance'] >= th_a).sum()) if not all_sc.empty else '0'
    m_ents  = str(all_sc['entity_id'].nunique()) if not all_sc.empty else '0'

    table_data = []
    if not all_sc.empty:
        top = all_sc.nlargest(20, 'dissonance')
        for _, r in top.iterrows():
            flag = '⚠️' if r['dissonance'] >= th_a else '✅'
            table_data.append([
                r['entity_id'], str(r['window_start'])[:16],
                f"{r['dissonance']:.3f}", flag, r['source']
            ])

    return (fig_hm, fig_tr, report, m_mean, m_max, m_alert, m_ents, table_data)


DEFAULT_JSON_UI = json.dumps({
    'org_name':  CFG['org_name'],
    'bimester':  BIMESTER,
    'thresholds': CFG['thresholds'],
    'nodes':     CFG.get('nodes', [])
}, indent=2)


with gr.Blocks(
    title='GSL Modo 1 — Observación Pasiva',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate'),
) as demo:

    gr.Markdown(f"""
# 🛡️ GSL Modo 1 — Observación Pasiva
**{CFG['org_name']}** · Bimestre {BIMESTER}

Sistema de detección por firma geométrica. Solo observa — no interviene en ningún sistema.
""")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown('### ⚙️ Configuración')
            config_box = gr.Code(value=DEFAULT_JSON_UI, language='json',
                                 label='org_config (editable)', lines=14)
            with gr.Row():
                siem_platform = gr.Dropdown(
                    choices=['chronicle','splunk','sentinel','cef','csv'],
                    value='chronicle', label='Plataforma SIEM (mock)')
                seed_sl = gr.Slider(1, 999, value=42, step=1, label='Semilla')
            siem_file = gr.Textbox(
                placeholder='ruta/al/siem_export.json  (vacío = mock)',
                label='Archivo SIEM (Nivel 0) — opcional')
            run_btn = gr.Button('▶ Ejecutar análisis', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('### 📊 Métricas bimestrales')
            with gr.Row():
                m_mean  = gr.Textbox(label='Disonancia media',    interactive=False)
                m_max   = gr.Textbox(label='Disonancia máxima',   interactive=False)
            with gr.Row():
                m_alert = gr.Textbox(label='Ventanas ≥ umbral',   interactive=False)
                m_ents  = gr.Textbox(label='Extremos activos',    interactive=False)
            gr.Markdown("""
### 🔍 Niveles de despliegue SIEM
**Nivel 0** — Archivo exportado del SIEM  
→ Coloca el archivo y especifica la ruta arriba

**Nivel 1** — API polling  
→ Agrega `api_endpoint` y `api_key` al config

**Nivel 2** — Stream / webhook  
→ Agrega `stream_buffer_path` al config
""")

    gr.Markdown('### 🗺️ Mapa de disonancia')
    fig_hm_out = gr.Plot()

    gr.Markdown('### 📈 Tendencia bimestral')
    fig_tr_out = gr.Plot()

    gr.Markdown('### 📋 Top 20 ventanas de mayor disonancia')
    table_out = gr.Dataframe(
        headers=['Extremo','Ventana','Disonancia','Estado','Fuente'],
        wrap=True)

    gr.Markdown('### 📄 Reporte bimestral')
    report_out = gr.Markdown()

    with gr.Accordion('📚 Guía de uso', open=False):
        gr.Markdown("""
## Flujo de adopción

| Paso | Acción | Resultado |
|---|---|---|
| 1 | Ejecutar con mock=True | Verificar que el pipeline funciona con datos sintéticos |
| 2 | Cargar `org_config.json` real | Adaptar usuarios, activos y umbrales a la organización |
| 3 | Nivel 0: exportar reporte del SIEM | Primer análisis con datos reales |
| 4 | Revisar reporte bimestral | Validar precisión — ¿los extremos marcados son genuinamente anómalos? |
| 5 | Nivel 1: agregar API key | Análisis casi en tiempo real |
| 6 | Si precisión satisfactoria | Considerar activar **Modo 2** para extremos validados |

## Formatos SIEM soportados
- **Chronicle UDM** — campos `metadata`, `principal`, `target`, `security_result`
- **Splunk JSON** — campos `_time`, `user`, `src`, `dest`, `EventCode`
- **Microsoft Sentinel** — campos `TimeGenerated`, `Account`, `OperationName`
- **CEF genérico** — líneas `CEF:0|vendor|product|...`
- **CSV genérico** — detección automática de columnas comunes
""")

    run_btn.click(
        fn=refresh_analysis,
        inputs=[config_box, siem_file, siem_platform, seed_sl],
        outputs=[fig_hm_out, fig_tr_out, report_out,
                 m_mean, m_max, m_alert, m_ents, table_out]
    )

print('✅ Dashboard listo. Lanzando...')
demo.launch(share=False, inbrowser=True)

## Puertos de integración (salida hacia módulos posteriores)

| Puerto | Artefacto | Destino |
|---|---|---|
| P1 | `principal_normalized_bundle.csv` | Módulo de dominio / análisis adicional |
| P2 | `siem_*_normalized_bundle.csv` | Módulo SIEM / correlación |
| P3 | `manifold_principal.json` | Modo 2 — base del PolicyAdapter |
| P4 | `manifold_siem_*.json` | Modo 2 — manifold del nodo SIEM |
| P5 | `dissonance_scores.csv` | Panel del cliente / SIEM del cliente |
| P6 | `bimester_report_*.md` | Reporte ejecutivo / cumplimiento |

Este notebook termina aquí. El Modo 2 consume los manifolds de P3 y P4
para ejecutar respuestas adaptativas cuando el cliente lo autoriza.

In [ ]:
# ─── ExportPackage final ──────────────────────────────────────
import zipfile

export_zip = ARTIFACTS.parent / f'gsl_m1_{CFG["principal_id"]}_{BIMESTER}.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ARTIFACTS.iterdir():
        zf.write(f, arcname=f.name)

manifest = {
    'run_id':       RUN_ID,
    'bimester':     BIMESTER,
    'org_name':     CFG['org_name'],
    'principal_id': CFG['principal_id'],
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'mode':         'GSL-Modo1-Pasivo',
    'artifacts':    [f.name for f in ARTIFACTS.iterdir()],
    'siem_nodes':   [n['node_id'] for n in SIEM_NODES],
    'scores_summary': {
        'total_windows':    len(all_scores),
        'mean_dissonance':  round(all_scores['dissonance'].mean(), 4) if not all_scores.empty else 0,
        'max_dissonance':   round(all_scores['dissonance'].max(), 4)  if not all_scores.empty else 0,
        'windows_over_alert': int((all_scores['dissonance'] >= th_alert).sum())
                              if not all_scores.empty else 0,
    }
}
with open(ARTIFACTS / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\n✅ ExportPackage: {export_zip}')
print(f'   Artefactos incluidos: {len(manifest["artifacts"])}')
print(f'   Tamaño: {export_zip.stat().st_size / 1024:.1f} KB')
print(f'\n📋 Manifiesto:')
print(json.dumps(manifest['scores_summary'], indent=2))